# MOTIF Deep Dive

Detailed analysis of parameter calibration and correlation-based biomarker discovery.

In [ ]:
import sys
sys.path.insert(0, '..')
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from src.synthetic_data import generate_cohort
from src.motif_pipeline import calibrate_patient, generate_motif_proxies, extract_features

## Generate and calibrate a 30-patient cohort

For detailed analysis, calibrate first 10 patients (to keep runtime reasonable).

In [ ]:
print('Generating 30-patient cohort...')
cohort = generate_cohort(N_resolution=10, N_chronic=10, N_death=10, seed=42, verbose=False)
print(f'Generated {len(cohort)} patients')
print('\nCalibrating first 10 patients for analysis...')
results = []
for i, patient in enumerate(cohort[:10]):
    print(f'  Patient {i+1}/10...', end='\r')
    fitted_params, fitted_P0, fit_quality = calibrate_patient(patient, n_restarts=1, verbose=False)
    proxies = generate_motif_proxies(patient, fitted_params, fitted_P0)
    features = extract_features(patient, proxies)
    results.append({
        'patient_id': patient['id'],
        'outcome': patient['outcome'],
        'fit_quality': fit_quality,
        'fitted_params': fitted_params,
        'fitted_P0': fitted_P0,
        'proxies': proxies,
        'features': features
    })
print('\nCalibration complete')

## Calibration quality histogram

Plot R-squared values across calibrated patients.

In [ ]:
fig, ax = plt.subplots(figsize=(8, 5))
fit_qualities = [r['fit_quality'] for r in results]
ax.hist(fit_qualities, bins=8, color='steelblue', edgecolor='black', alpha=0.7)
ax.axvline(np.mean(fit_qualities), color='red', linestyle='--', linewidth=2, label=f'Mean = {np.mean(fit_qualities):.3f}')
ax.axvline(np.median(fit_qualities), color='green', linestyle='--', linewidth=2, label=f'Median = {np.median(fit_qualities):.3f}')
ax.set_xlabel('Fit Quality (R-squared)', fontsize=12)
ax.set_ylabel('Number of Patients', fontsize=12)
ax.set_title('MOTIF Calibration Quality Distribution', fontsize=14, fontweight='bold')
ax.legend(fontsize=10)
ax.grid(True, alpha=0.3, axis='y')
plt.tight_layout()
plt.show()
print(f'Mean R2: {np.mean(fit_qualities):.4f}')
print(f'Std R2: {np.std(fit_qualities):.4f}')

## Feature extraction and summary

Extract features from calibrated patients.

In [ ]:
import pandas as pd

# Combine all features
features_list = []
for r in results:
    row = {'patient_id': r['patient_id'], 'outcome': r['outcome'], 'R2': r['fit_quality']}
    row.update(r['features'])
    features_list.append(row)

features_df = pd.DataFrame(features_list)
print('Feature Summary:')
print(features_df.head(10))
print(f'\nShape: {features_df.shape}')
print(f'Columns: {list(features_df.columns)}')

## Next Steps

The MOTIF pipeline has successfully:
1. Calibrated ODE parameters to observed multiomics data
2. Generated proxy trajectories for latent variables
3. Extracted interpretable features

These features can now be used for:
- Outcome prediction via supervised learning
- Biomarker discovery via correlation analysis
- Clinical interpretation

See the next notebooks for UDE training and results comparison.